# Kelly Criterion & Bankroll Management (EXTRA)

**Chapter 9: From Predictions to Profit: Navigating the World of Soccer Betting**

## Learning Objectives

- Understand the Kelly Criterion formula
- Calculate optimal bet sizes
- Implement Kelly betting strategy
- Compare Kelly vs fixed stake betting
- Understand fractional Kelly (risk management)
- Implement proper bankroll management
- Avoid risk of ruin

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

np.random.seed(42)

print("Kelly Criterion & Bankroll Management Toolkit Ready!")

## Introduction

The **Kelly Criterion** is a mathematical formula for optimal bet sizing that:
- Maximizes long-term growth
- Minimizes risk of ruin
- Accounts for edge and odds

### The Problem

With fixed stakes:
- Bet too much → Risk of ruin
- Bet too little → Slow growth

Kelly solves this by calculating the **optimal fraction** of bankroll to bet.

---

# 1. The Kelly Criterion Formula

## Formula

```
f* = (p × b - q) / b
```

Where:
- **f*** = Optimal fraction of bankroll to bet
- **p** = Probability of winning
- **q** = Probability of losing (1 - p)
- **b** = Net odds (decimal odds - 1)

### For decimal odds:

```
f* = (p × (odds - 1) - (1 - p)) / (odds - 1)
```

In [ ]:
def kelly_criterion(probability, odds):
    """
    Calculate Kelly Criterion bet size.
    
    Args:
        probability: Your estimated win probability (0-1)
        odds: Decimal odds
    
    Returns:
        fraction: Optimal fraction of bankroll to bet
    """
    p = probability
    q = 1 - p
    b = odds - 1  # Net odds
    
    # Kelly formula
    fraction = (p * b - q) / b
    
    # Never bet negative amount
    fraction = max(fraction, 0)
    
    return fraction

# Example
prob = 0.55  # 55% chance of winning
odds = 2.00  # Even odds

fraction = kelly_criterion(prob, odds)

print(f"Kelly Criterion Calculation:\n")
print(f"Win Probability: {prob:.1%}")
print(f"Odds: {odds:.2f}")
print(f"Optimal Bet Size: {fraction:.1%} of bankroll")

# Example with $1000 bankroll
bankroll = 1000
bet_size = bankroll * fraction
print(f"\nWith ${bankroll} bankroll:")
print(f"Bet Size: ${bet_size:.2f}")

## 1.1 Visualizing Kelly Bet Size

In [ ]:
# Create heatmap of Kelly fractions
probs = np.linspace(0.4, 0.7, 50)
odds_range = np.linspace(1.5, 4.0, 50)

kelly_matrix = np.zeros((len(probs), len(odds_range)))

for i, p in enumerate(probs):
    for j, o in enumerate(odds_range):
        kelly_matrix[i, j] = kelly_criterion(p, o)

fig, ax = plt.subplots(figsize=(10, 8))

im = ax.imshow(kelly_matrix, aspect='auto', origin='lower', cmap='RdYlGn',
               extent=[odds_range.min(), odds_range.max(), probs.min(), probs.max()])

ax.set_xlabel('Decimal Odds', fontsize=12)
ax.set_ylabel('Win Probability', fontsize=12)
ax.set_title('Kelly Criterion Bet Size (% of Bankroll)', fontsize=14, fontweight='bold')

cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Bet Fraction', fontsize=11)

# Add contour lines
contours = ax.contour(odds_range, probs, kelly_matrix, colors='black', alpha=0.3, linewidths=0.5)
ax.clabel(contours, inline=True, fontsize=8)

plt.tight_layout()
plt.show()

print("Key Insight: Higher probability + higher odds = larger bet size!")

## 1.2 When NOT to Bet (Kelly = 0)

Kelly tells you to bet **nothing** when you have no edge.

In [ ]:
# Examples where Kelly = 0 or negative
examples = [
    {'prob': 0.50, 'odds': 2.00, 'description': 'Fair odds, no edge'},
    {'prob': 0.45, 'odds': 2.00, 'description': 'Negative edge'},
    {'prob': 0.30, 'odds': 3.00, 'description': 'Implied 33%, you think 30%'},
    {'prob': 0.60, 'odds': 1.60, 'description': 'Implied 62.5%, you think 60%'},
]

print("When Kelly Says DON'T BET:\n")
for ex in examples:
    fraction = kelly_criterion(ex['prob'], ex['odds'])
    implied_prob = 1 / ex['odds']
    edge = ex['prob'] - implied_prob
    
    print(f"{ex['description']}")
    print(f"  Your Prob: {ex['prob']:.1%}, Odds: {ex['odds']:.2f}, Implied: {implied_prob:.1%}")
    print(f"  Edge: {edge:+.1%}")
    print(f"  Kelly: {fraction:.1%}")
    print(f"  {'❌ DO NOT BET' if fraction <= 0 else '✅ Bet'}\n")

---

# 2. Implementing Kelly Betting Strategy

Let's simulate a full season using Kelly Criterion.

In [ ]:
# Generate season data
np.random.seed(42)

num_matches = 380
matches = []

for i in range(num_matches):
    # True probabilities
    home_prob_true = np.random.uniform(0.35, 0.55)
    draw_prob_true = np.random.uniform(0.20, 0.30)
    away_prob_true = 1 - home_prob_true - draw_prob_true
    
    # Bookmaker odds (with 5% margin)
    overround = 1.05
    home_odds = (1 / home_prob_true) * overround
    draw_odds = (1 / draw_prob_true) * overround
    away_odds = (1 / away_prob_true) * overround
    
    # Your predictions (sometimes better, sometimes worse)
    noise = np.random.normal(0, 0.05)
    home_prob_yours = np.clip(home_prob_true + noise, 0.1, 0.9)
    draw_prob_yours = np.clip(draw_prob_true + noise, 0.1, 0.9)
    away_prob_yours = 1 - home_prob_yours - draw_prob_yours
    
    # Actual outcome
    outcome_rand = np.random.random()
    if outcome_rand < home_prob_true:
        result = 'H'
    elif outcome_rand < home_prob_true + draw_prob_true:
        result = 'D'
    else:
        result = 'A'
    
    matches.append({
        'match_id': i + 1,
        'home_odds': home_odds,
        'draw_odds': draw_odds,
        'away_odds': away_odds,
        'your_home_prob': home_prob_yours,
        'your_draw_prob': draw_prob_yours,
        'your_away_prob': away_prob_yours,
        'result': result
    })

matches_df = pd.DataFrame(matches)
print(f"Generated {len(matches_df)} matches for Kelly simulation")

In [ ]:
def kelly_betting_strategy(matches_df, initial_bankroll=1000, kelly_fraction=1.0):
    """
    Simulate Kelly Criterion betting strategy.
    
    Args:
        matches_df: DataFrame with matches and predictions
        initial_bankroll: Starting bankroll
        kelly_fraction: Fraction of Kelly to use (1.0 = full Kelly, 0.5 = half Kelly)
    
    Returns:
        results_df: DataFrame with bet-by-bet results
    """
    bankroll = initial_bankroll
    results = []
    
    for _, match in matches_df.iterrows():
        # Find best betting opportunity
        opportunities = [
            ('H', match['your_home_prob'], match['home_odds']),
            ('D', match['your_draw_prob'], match['draw_odds']),
            ('A', match['your_away_prob'], match['away_odds'])
        ]
        
        # Calculate Kelly for each outcome
        best_bet = None
        best_kelly = 0
        
        for outcome, prob, odds in opportunities:
            kelly_frac = kelly_criterion(prob, odds)
            if kelly_frac > best_kelly:
                best_kelly = kelly_frac
                best_bet = (outcome, prob, odds)
        
        # Place bet if Kelly > 0
        if best_kelly > 0 and bankroll > 0:
            outcome, prob, odds = best_bet
            bet_size = bankroll * best_kelly * kelly_fraction
            bet_size = min(bet_size, bankroll)  # Never bet more than bankroll
            
            # Determine profit/loss
            if match['result'] == outcome:
                profit = bet_size * (odds - 1)
            else:
                profit = -bet_size
            
            bankroll += profit
            
            results.append({
                'match_id': match['match_id'],
                'prediction': outcome,
                'actual': match['result'],
                'odds': odds,
                'kelly_fraction': best_kelly,
                'bet_size': bet_size,
                'profit': profit,
                'bankroll': bankroll,
                'correct': match['result'] == outcome
            })
    
    return pd.DataFrame(results)

# Run Kelly strategy
results_kelly = kelly_betting_strategy(matches_df, initial_bankroll=1000, kelly_fraction=1.0)

print(f"Kelly Betting Results:\n")
print(f"Initial Bankroll: $1,000")
print(f"Final Bankroll: ${results_kelly['bankroll'].iloc[-1]:,.2f}")
print(f"Total Profit: ${results_kelly['bankroll'].iloc[-1] - 1000:,.2f}")
print(f"ROI: {((results_kelly['bankroll'].iloc[-1] - 1000) / 1000) * 100:+.1f}%")
print(f"\nBets Placed: {len(results_kelly)}")
print(f"Win Rate: {results_kelly['correct'].mean():.1%}")

## 2.1 Compare Kelly vs Fixed Stake

In [ ]:
# Fixed stake strategy for comparison
def fixed_stake_strategy(matches_df, stake=10):
    """
    Fixed stake betting on best opportunity.
    """
    bankroll = 0
    results = []
    
    for _, match in matches_df.iterrows():
        # Find best opportunity by expected value
        opportunities = [
            ('H', match['your_home_prob'], match['home_odds']),
            ('D', match['your_draw_prob'], match['draw_odds']),
            ('A', match['your_away_prob'], match['away_odds'])
        ]
        
        best_ev = -float('inf')
        best_bet = None
        
        for outcome, prob, odds in opportunities:
            ev = prob * (odds - 1) - (1 - prob)
            if ev > best_ev:
                best_ev = ev
                best_bet = (outcome, prob, odds)
        
        # Only bet if positive EV
        if best_ev > 0:
            outcome, prob, odds = best_bet
            
            if match['result'] == outcome:
                profit = stake * (odds - 1)
            else:
                profit = -stake
            
            bankroll += profit
            
            results.append({
                'match_id': match['match_id'],
                'prediction': outcome,
                'actual': match['result'],
                'odds': odds,
                'bet_size': stake,
                'profit': profit,
                'bankroll': 1000 + bankroll,  # Start from 1000
                'correct': match['result'] == outcome
            })
    
    return pd.DataFrame(results)

results_fixed = fixed_stake_strategy(matches_df, stake=10)

print(f"Fixed Stake Results:\n")
print(f"Final Bankroll: ${results_fixed['bankroll'].iloc[-1]:,.2f}")
print(f"Total Profit: ${results_fixed['bankroll'].iloc[-1] - 1000:,.2f}")
print(f"ROI: {((results_fixed['bankroll'].iloc[-1] - 1000) / 1000) * 100:+.1f}%")

In [ ]:
# Plot comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Bankroll evolution
ax1.plot(results_kelly['match_id'], results_kelly['bankroll'], 
         linewidth=2, label='Kelly Criterion', alpha=0.8)
ax1.plot(results_fixed['match_id'], results_fixed['bankroll'], 
         linewidth=2, label='Fixed Stake ($10)', alpha=0.8)
ax1.axhline(1000, color='black', linestyle='--', alpha=0.5, label='Initial Bankroll')

ax1.set_xlabel('Match Number', fontsize=12)
ax1.set_ylabel('Bankroll ($)', fontsize=12)
ax1.set_title('Bankroll Evolution: Kelly vs Fixed', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Right: Bet sizes over time (Kelly only)
ax2.scatter(results_kelly['match_id'], results_kelly['bet_size'], 
            c=results_kelly['correct'], cmap='RdYlGn', alpha=0.6, s=20)
ax2.set_xlabel('Match Number', fontsize=12)
ax2.set_ylabel('Bet Size ($)', fontsize=12)
ax2.set_title('Kelly Bet Sizes (Green=Win, Red=Loss)', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Notice: Kelly bet sizes vary with bankroll and edge!")

---

# 3. Fractional Kelly (Risk Management)

**Full Kelly** can be aggressive. Many professionals use **fractional Kelly**:

- **Full Kelly** (f = 1.0): Maximum growth, high variance
- **Half Kelly** (f = 0.5): 75% of growth, 50% of variance
- **Quarter Kelly** (f = 0.25): Conservative, very safe

### Why Fractional Kelly?

1. **Estimation errors**: Your probabilities might be wrong
2. **Risk tolerance**: Reduce volatility
3. **Psychological comfort**: Easier to stick with

In [ ]:
# Compare different Kelly fractions
kelly_fractions = [0.25, 0.5, 0.75, 1.0]
results_by_fraction = {}

for frac in kelly_fractions:
    results = kelly_betting_strategy(matches_df, initial_bankroll=1000, kelly_fraction=frac)
    results_by_fraction[frac] = results

# Plot comparison
fig, ax = plt.subplots(figsize=(12, 6))

for frac, results in results_by_fraction.items():
    ax.plot(results['match_id'], results['bankroll'], 
            linewidth=2, label=f'{frac:.0%} Kelly', alpha=0.8)

ax.axhline(1000, color='black', linestyle='--', alpha=0.5, label='Initial Bankroll')
ax.set_xlabel('Match Number', fontsize=12)
ax.set_ylabel('Bankroll ($)', fontsize=12)
ax.set_title('Fractional Kelly Comparison', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print summary
print("Fractional Kelly Summary:\n")
for frac, results in results_by_fraction.items():
    final = results['bankroll'].iloc[-1]
    profit = final - 1000
    roi = (profit / 1000) * 100
    max_drawdown = (results['bankroll'].cummax() - results['bankroll']).max()
    
    print(f"{frac:.0%} Kelly:")
    print(f"  Final Bankroll: ${final:,.2f}")
    print(f"  ROI: {roi:+.1f}%")
    print(f"  Max Drawdown: ${max_drawdown:,.2f}")
    print()

## Summary

In this notebook, you learned:

✅ **Kelly Criterion formula** - Optimal bet sizing

✅ **Implementation** - Applying Kelly to real bets

✅ **Kelly vs Fixed** - Comparison of strategies

✅ **Fractional Kelly** - Risk management through position sizing

✅ **Bankroll management** - Protecting against ruin

### Key Takeaways

**1. Kelly is mathematically optimal**
- Maximizes long-term growth
- Minimizes risk of ruin
- Automatically adjusts bet size

**2. Full Kelly can be aggressive**
- High variance
- Requires accurate probabilities
- Psychologically difficult

**3. Fractional Kelly is practical**
- Half Kelly recommended for most
- Reduces variance significantly
- More robust to estimation errors

**4. Bankroll management is crucial**
- Never risk entire bankroll
- Bet size should scale with edge
- Discipline is essential

## Practice Exercises

1. **Calculate Kelly**: For 60% win probability at 2.5 odds, what's optimal bet size?

2. **Compare fractions**: Simulate 1000 bets with different Kelly fractions

3. **Estimation error**: How does 5% probability error affect Kelly performance?

4. **Risk of ruin**: Calculate probability of losing 50% of bankroll with full vs half Kelly

5. **Optimal fraction**: Find the Kelly fraction that maximizes Sharpe ratio

6. **Multiple bets**: How to apply Kelly when betting on multiple matches simultaneously?

7. **Stop loss**: Implement a stop-loss rule with Kelly betting

8. **Comparison**: Compare Kelly to martingale and other betting systems